## Sportmonks Germany 201 — 01: Fixture corpus

Build the match list for **Germany** under the competitive calendar: **Euro 2024** (`league_id` 1326), **2025 UEFA Nations League** (1538), and **WC Qualification Europe** (720). Keep **2024–2025** kickoffs only and **finished** matches (`state_id` 5). Friendlies are excluded by competition whitelist.

**Run Jupyter from the repository root** so `load_dotenv(".env")` matches the other Sportmonks notebooks.

In [1]:
import json
import os
import sys
from datetime import date
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd()
if (ROOT / "sportmonks").is_dir():
    SPORTMONKS_DIR = ROOT / "sportmonks"
    sys.path.insert(0, str(SPORTMONKS_DIR))
else:
    SPORTMONKS_DIR = ROOT
    sys.path.insert(0, str(ROOT))

import sportmonks_germany_api as smg

In [2]:
load_dotenv(".env")

API_TOKEN = os.getenv("API_TOKEN")

### Resolve Germany `team_id`
Set `GERMANY_TEAM_ID_OVERRIDE` if `/teams/search/Germany` returns multiple clubs and the heuristic picks the wrong row.

In [3]:
GERMANY_TEAM_ID_OVERRIDE = None  # e.g. 12345

team = None
hits = []
if GERMANY_TEAM_ID_OVERRIDE is not None:
    GERMANY_TEAM_ID = int(GERMANY_TEAM_ID_OVERRIDE)
else:
    hits = smg.search_teams(API_TOKEN, "Germany")

pd.DataFrame(hits).head(20) if hits else None

,id,sport_id,country_id,venue_id,gender,name,short_code,image_path,founded,type,placeholder,last_played_at
0,18660,1,11,1944.0,male,Germany,GER,https://cdn.sportmonks.com/images/soccer/teams...,1900.0,national,False,2026-03-30 18:45:00
1,63338,1,11,NaN,male,Germany U23,GER U23,https://cdn.sportmonks.com/images/soccer/teams...,NaN,national,False,None


In [4]:
if GERMANY_TEAM_ID_OVERRIDE is None:
    team = smg.pick_germany_national_team(hits)
    if team is None:
        raise RuntimeError("No team from search; set GERMANY_TEAM_ID_OVERRIDE")
    GERMANY_TEAM_ID = int(team["id"])

GERMANY_TEAM_ID, (team or {}).get("name")

(18660, 'Germany')

In [5]:
meta = {
    "germany_team_id": GERMANY_TEAM_ID,
    "team_name": (team or {}).get("name"),
    "league_ids": sorted(smg.LEAGUE_IDS),
}
(SPORTMONKS_DIR / "germany_201_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
meta

{'germany_team_id': 18660,
 'team_name': 'Germany',
 'league_ids': [720, 1326, 1538]}

**European Championship:** `1326`  
**WC Qualification Europe:** `720`  
**UEFA Nations League:** `1538`

### Ingest: chunked `fixtures/between/{start}/{end}/{team_id}`
Sportmonks limits each between-query window (use ≤100 days per chunk).

In [6]:
start = date(2024, 1, 1)
end = date(2025, 12, 31)

all_fixtures = []
for a, b in smg.date_chunks(start, end, max_days=100):
    chunk = smg.fetch_fixtures_between_team(
        API_TOKEN, GERMANY_TEAM_ID, a.isoformat(), b.isoformat()
    )
    all_fixtures.extend(chunk)

len(all_fixtures)

21

In [7]:
filtered = smg.filter_germany_corpus(all_fixtures)
df = pd.json_normalize(filtered).drop_duplicates(subset=["id"], keep="first")
df["starting_at"] = pd.to_datetime(df["starting_at"], errors="coerce")
df = df.sort_values("starting_at")
df[["id", "league_id", "season_id", "name", "starting_at", "state_id"]]

,id,league_id,season_id,name,starting_at,state_id
0,19032613,1326,22842,Germany vs Scotland,2024-06-14 19:00:00,5
1,19032614,1326,22842,Germany vs Hungary,2024-06-19 16:00:00,5
2,19032615,1326,22842,Switzerland vs Germany,2024-06-23 19:00:00,5
3,19032605,1326,22842,Germany vs Denmark,2024-06-29 19:00:00,5
4,19032601,1326,22842,Spain vs Germany,2024-07-05 16:00:00,7
5,19085361,1538,23188,Germany vs Hungary,2024-09-07 18:45:00,5
6,19085364,1538,23188,Netherlands vs Germany,2024-09-10 18:45:00,5
7,19085365,1538,23188,Bosnia and Herzegovina vs Germany,2024-10-11 18:45:00,5
8,19085368,1538,23188,Germany vs Netherlands,2024-10-14 18:45:00,5
9,19085369,1538,23188,Germany vs Bosnia and Herzegovina,2024-11-16 19:45:00,5


In [8]:
summary = df.groupby("league_id").size().rename("matches").sort_index()
summary

league_id
720      6
1326     5
1538    10
Name: matches, dtype: int64

In [9]:
out_csv = SPORTMONKS_DIR / "fixtures_germany_201.csv"
df.to_csv(out_csv, index=False)

### Nagelsmann framing
Julian Nagelsmann took over the Germany senior men’s team before this window; competitive matches in **2024–2025** in the three listed tournaments are interpreted as his-era games for this pipeline (no extra date filter).